In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/csiro-biomass/sample_submission.csv
/kaggle/input/csiro-biomass/train.csv
/kaggle/input/csiro-biomass/test.csv
/kaggle/input/csiro-biomass/test/ID1001187975.jpg
/kaggle/input/csiro-biomass/train/ID2099464826.jpg
/kaggle/input/csiro-biomass/train/ID2037861084.jpg
/kaggle/input/csiro-biomass/train/ID1211362607.jpg
/kaggle/input/csiro-biomass/train/ID1853508321.jpg
/kaggle/input/csiro-biomass/train/ID193102215.jpg
/kaggle/input/csiro-biomass/train/ID698608346.jpg
/kaggle/input/csiro-biomass/train/ID1859251563.jpg
/kaggle/input/csiro-biomass/train/ID1880764911.jpg
/kaggle/input/csiro-biomass/train/ID853954911.jpg
/kaggle/input/csiro-biomass/train/ID1403107574.jpg
/kaggle/input/csiro-biomass/train/ID1781353117.jpg
/kaggle/input/csiro-biomass/train/ID384648061.jpg
/kaggle/input/csiro-biomass/train/ID1563418511.jpg
/kaggle/input/csiro-biomass/train/ID2125100696.jpg
/kaggle/input/csiro-biomass/train/ID482555369.jpg
/kaggle/input/csiro-biomass/train/ID638711343.jpg
/kaggle/input/c

In [2]:
# ============================================================================
# CELL 1: Setup and Data Loading
# ============================================================================
import numpy as np
import pandas as pd
import os
import torch
import torchvision
import torch.nn as nn
import pytorch_lightning as pl
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import v2
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pytorch_lightning import Trainer
from tqdm import tqdm
import cv2
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# Load data
train_file_path = "/kaggle/input/csiro-biomass/train.csv"
test_file_path = "/kaggle/input/csiro-biomass/test.csv"

train_pd = pd.read_csv(train_file_path)
test_pd_original = pd.read_csv(test_file_path)  # Keep original for submission
test_pd = test_pd_original.copy()  # Working copy for predictions

print(f"Train shape: {train_pd.shape}")
print(f"Test shape: {test_pd.shape}")

Train shape: (1785, 9)
Test shape: (5, 3)


In [3]:
# ============================================================================
# CELL 2: HEIGHT PREDICTION
# ============================================================================
print("\n=== STEP 1: Height Prediction ===")

height_model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None, weights_backbone=None)
path = "/kaggle/input/csiro-biomass"
local_weights = "/kaggle/input/mask-rcnn-models/pytorch/default/10/Maskrcnn_best.pt"

try:
    state_dict = torch.load(local_weights, map_location="cpu", weights_only=False)
    height_model = state_dict
    height_model.eval()
    
    for index, image_path in test_pd["image_path"].items():
        image_path = os.path.join(path, image_path)
        image = Image.open(image_path).convert("RGB")
        
        image_transform = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        image_tensor = image_transform(image)
        
        with torch.no_grad():
            outputs = height_model([image_tensor])
        
        output_pixels = outputs[0]["boxes"].cpu().numpy()
        x1, y1, x2, y2 = output_pixels[0]
        image_in_cm = y2 / 2.54
        test_pd.loc[index, "Height_Ave_cm"] = image_in_cm
    
    print("✅ Height predictions completed using Mask R-CNN")
except:
    # Fallback to mean height
    mean_height = train_pd["Height_Ave_cm"].mean()
    test_pd["Height_Ave_cm"] = mean_height
    print(f"✅ Height predictions using mean: {mean_height:.2f}")


=== STEP 1: Height Prediction ===
✅ Height predictions completed using Mask R-CNN


species model

In [4]:
import torch

torch.cuda.empty_cache()

In [5]:
# ============================================================================
# CELL 3: SPECIES PREDICTION (EFFICIENTNET-B6) - IMPROVED
# ============================================================================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from torchvision.transforms import v2
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm
from collections import Counter

print("\n=== STEP 2: Species Prediction (EfficientNet-B6) [Improved] ===")

# --- Global Encoders ---
SPECIES_LE = LabelEncoder()
TARGET_LE = LabelEncoder()

# Fit encoders
SPECIES_LE.fit(train_pd["Species"].astype(str).unique())
TARGET_LE.fit(train_pd["target_name"].astype(str).unique())

def safe_encode(le, val):
    val_str = str(val)
    if val_str in le.classes_:
        return le.transform([val_str])[0]
    return 0

# --- Dataset Class ---
class SpeciesDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        tabular = torch.tensor([
            float(row["target_name"]), 
            float(row["Height_Ave_cm"])
        ], dtype=torch.float32)
        
        y = torch.tensor(int(row["Species"]), dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
        return image, tabular, y

# --- Data Module ---
class SpeciesDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch_size=8, num_workers=2):
        super().__init__()
        self.train_df, self.valid_df = train_df.copy(), valid_df.copy()
        self.root_dir, self.batch_size, self.num_workers = root_dir, batch_size, num_workers

    def setup(self, stage=None):
        # 1. Encode Data
        for df in [self.train_df, self.valid_df]:
            if not np.issubdtype(df["Species"].dtype, np.number):
                df["Species"] = df["Species"].apply(lambda x: safe_encode(SPECIES_LE, x))
            if not np.issubdtype(df["target_name"].dtype, np.number):
                df["target_name"] = df["target_name"].apply(lambda x: safe_encode(TARGET_LE, x))

        # 2. Calculate Class Weights for Sampler (Fix for Imbalanced Data)
        class_counts = Counter(self.train_df["Species"])
        num_samples = len(self.train_df)
        class_weights = {c: num_samples / count for c, count in class_counts.items()}
        self.sample_weights = [class_weights[t] for t in self.train_df["Species"]]

        # 3. Improved Transforms (Added Rotation & ColorJitter)
        self.train_tf = v2.Compose([
            v2.RandomResizedCrop(380, scale=(0.7, 1.0)), # B6 input size
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.3),   # Plants can be viewed from any top-down angle
            v2.RandomRotation(degrees=30),  # IMPORTANT: Helps with orientation invariance
            v2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05), # Lighting invariance
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        self.valid_tf = v2.Compose([
            v2.Resize(400), 
            v2.CenterCrop(380), 
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        self.train_ds = SpeciesDataset(self.train_df, self.root_dir, self.train_tf)
        self.valid_ds = SpeciesDataset(self.valid_df, self.root_dir, self.valid_tf)

    def train_dataloader(self):
        # Using WeightedRandomSampler to handle class imbalance
        sampler = WeightedRandomSampler(
            weights=self.sample_weights,
            num_samples=len(self.sample_weights),
            replacement=True
        )
        return DataLoader(self.train_ds, batch_size=self.batch_size, sampler=sampler, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

# --- Classifier Model (EfficientNet-B6) ---
class SpeciesClassifier(pl.LightningModule):
    def __init__(self, num_classes, target_dim, learning_rate=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.lr = learning_rate
        
        # 1. Initialize Architecture
        print("Initializing EfficientNet-B6...")
        self.base_model = models.efficientnet_b6(weights=None)
        
        # 2. Load Weights
        weights_path = "/kaggle/input/efficientnet-b6/pytorch/default/1/efficientnet_b6_lukemelas-24a108a5.pth"
        try:
            state_dict = torch.load(weights_path, weights_only=True)
            self.base_model.load_state_dict(state_dict, strict=False)
            print("✅ Weights loaded successfully.")
        except Exception as e:
            print(f"⚠️ Warning: Could not load weights: {e}")

        # 3. Get Feature Dimension
        self.img_dim = self.base_model.classifier[1].in_features 
        self.base_model.classifier = nn.Identity()

        # 4. Tabular & Fusion Heads
        self.target_emb = nn.Embedding(target_dim + 1, 16) # Increased dim slightly
        self.tabular_net = nn.Sequential(
            nn.Linear(16 + 1, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 512), # Increased hidden size
            nn.BatchNorm1d(512),
            nn.SiLU(), # Swish (SiLU) works well with EfficientNet
            nn.Dropout(0.5), # Higher dropout for regularization
            nn.Linear(512, num_classes)
        )
        
        # Label smoothing prevents overconfidence
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, img, target_name, height):
        img_feats = self.base_model(img)
        if len(img_feats.shape) > 2:
            img_feats = img_feats.view(img_feats.size(0), -1)
            
        t_feat = self.target_emb(target_name.long())
        tab_in = torch.cat([t_feat, height.unsqueeze(1)], dim=1)
        tab_feats = self.tabular_net(tab_in)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        # Differential learning rates: Slower for backbone, faster for head
        optimizer = torch.optim.AdamW([
            {'params': self.base_model.parameters(), 'lr': self.lr * 0.1},
            {'params': self.tabular_net.parameters(), 'lr': self.lr},
            {'params': self.head.parameters(), 'lr': self.lr}
        ], weight_decay=1e-2)
        
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, 
            max_lr=[self.lr * 0.1, self.lr, self.lr],
            total_steps=self.trainer.estimated_stepping_batches,
            pct_start=0.3,
            div_factor=25.0,
            final_div_factor=100.0
        )
        
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step"
            }
        }

# --- Training Section ---
image_root_dir = "/kaggle/input/csiro-biomass"

train_df, valid_df = train_test_split(
    train_pd, 
    test_size=0.15, # Slightly more data for training
    random_state=42, 
    stratify=train_pd["Species"]
)

# Batch size 8 + accumulate_grad_batches=2 simulates batch_size=16
datamodule = SpeciesDataModule(train_df=train_df, valid_df=valid_df, root_dir=image_root_dir, batch_size=8)
datamodule.setup()

num_species = len(SPECIES_LE.classes_)
target_count = len(TARGET_LE.classes_)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

species_model = SpeciesClassifier(
    num_classes=num_species, 
    target_dim=target_count,
    learning_rate=3e-4 # Slightly higher LR for OneCycle
).to(device)

early_stop_callback = EarlyStopping(monitor="val_loss", patience=5, mode="min", verbose=True)
checkpoint_callback = ModelCheckpoint(monitor="val_acc", save_top_k=1, mode="max", filename="best-species")
lr_monitor = LearningRateMonitor(logging_interval='step')

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=15,
    precision="16-mixed",
    accumulate_grad_batches=2, # Stability for small batch size
    callbacks=[early_stop_callback, checkpoint_callback, lr_monitor],
    gradient_clip_val=1.0
)

trainer.fit(species_model, datamodule)

# --- Inference Section (With TTA) ---
print("\n=== Starting Species Inference (With TTA) ===")
# Load best model
best_model_path = checkpoint_callback.best_model_path
if best_model_path:
    print(f"Loading best model from {best_model_path}")
    species_model = SpeciesClassifier.load_from_checkpoint(best_model_path)

species_model.eval()
species_model.to(device)

# Standard Transform
base_tf = v2.Compose([
    v2.Resize(400), v2.CenterCrop(380), v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

species_results = []

with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting Species"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img_pil = Image.open(img_path).convert("RGB")
            
            # --- TTA (Test Time Augmentation) ---
            # 1. Normal Image
            img_norm = base_tf(img_pil).unsqueeze(0).to(device)
            # 2. Flipped Image
            img_flip = base_tf(img_pil.transpose(Image.FLIP_LEFT_RIGHT)).unsqueeze(0).to(device)
            
            # Tabular Data
            t_idx = safe_encode(TARGET_LE, row["target_name"])
            tab_tensor = torch.tensor([[t_idx, float(row["Height_Ave_cm"])]], dtype=torch.float32).to(device)
            
            # Predict on both
            logits_norm = species_model(img_norm, tab_tensor[:, 0], tab_tensor[:, 1])
            logits_flip = species_model(img_flip, tab_tensor[:, 0], tab_tensor[:, 1])
            
            # Average probabilities
            probs = (F.softmax(logits_norm, dim=1) + F.softmax(logits_flip, dim=1)) / 2
            
            class_idx = probs.argmax(dim=1).item()
            actual_name = SPECIES_LE.inverse_transform([class_idx])[0]
            species_results.append(actual_name)
            
        except Exception as e:
            # Fallback to most common class if image fails
            species_results.append(SPECIES_LE.classes_[0])

test_pd["Species"] = species_results
print("✅ Species predictions completed & saved to test_pd")


=== STEP 2: Species Prediction (EfficientNet-B6) [Improved] ===
Initializing EfficientNet-B6...
✅ Weights loaded successfully.


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
2026-01-19 03:43:29.528717: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768794209.943156      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768794210.071716      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768794211.178743      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768794211.178775      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin

┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ base_model  │ EfficientNet     │ 40.7 M │ train │     0 │
│ 1 │ target_emb  │ Embedding        │     96 │ train │     0 │
│ 2 │ tabular_net │ Sequential       │  1.3 K │ train │     0 │
│ 3 │ head        │ Sequential       │  1.2 M │ train │     0 │
│ 4 │ loss_fn     │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 42.0 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 42.0 M                                                                                               
Total estimated model params size (MB): 167                                                                        
Modules in train mode: 920                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val_loss improved. New best score: 2.403
Metric val_loss improved by 0.546 >= min_delta = 0.0. New best score: 1.856
Metric val_loss improved by 0.530 >= min_delta = 0.0. New best score: 1.327
Metric val_loss improved by 0.152 >= min_delta = 0.0. New best score: 1.175
Metric val_loss improved by 0.165 >= min_delta = 0.0. New best score: 1.010
Metric val_loss improved by 0.083 >= min_delta = 0.0. New best score: 0.927
Metric val_loss improved by 0.048 >= min_delta = 0.0. New best score: 0.879
Metric val_loss improved by 0.050 >= min_delta = 0.0. New best score: 0.829
Metric val_loss improved by 0.019 >= min_delta = 0.0. New best score: 0.811
Metric val_loss improved by 0.039 >= min_delta = 0.0. New best score: 0.771
`Trainer.fit` stopped: `max_epochs=15` reached.



=== Starting Species Inference (With TTA) ===
Loading best model from /kaggle/working/lightning_logs/version_0/checkpoints/best-species.ckpt
Initializing EfficientNet-B6...
✅ Weights loaded successfully.


Predicting Species: 100%|██████████| 5/5 [00:00<00:00,  5.74it/s]

✅ Species predictions completed & saved to test_pd


In [6]:
import torch

torch.cuda.empty_cache()

In [7]:
# ============================================================================
# CELL 4: STATE PREDICTION (EFFICIENTNET-B6) - IMPROVED
# ============================================================================
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from torchvision.transforms import v2
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm
from collections import Counter

print("\n=== STEP 3: State Prediction (Custom Model: EfficientNet-B6) [Improved] ===")

# --- Global Encoder for State ---
STATE_LE = LabelEncoder()
STATE_LE.fit(train_pd["State"].astype(str).unique())
# Note: SPECIES_LE and TARGET_LE are reused from the previous cell

def safe_encode(le, val):
    val_str = str(val)
    if val_str in le.classes_:
        return le.transform([val_str])[0]
    return 0

# --- Dataset Class ---
class StateDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        # Tabular: [target_name, height, species]
        # We use the predicted 'Species' column (safe_encoded)
        tabular = torch.tensor([
            float(row["target_name"]), 
            float(row["Height_Ave_cm"]), 
            float(row["Species"]) 
        ], dtype=torch.float32)

        # Target Label: State
        # For test data, we put a dummy value (0) if State column doesn't exist
        y = torch.tensor(int(row["State"]) if "State" in row else 0, dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
        return image, tabular, y

# --- Data Module ---
class StateDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch_size=8, num_workers=2):
        super().__init__()
        self.train_df, self.valid_df = train_df.copy(), valid_df.copy()
        self.root_dir, self.batch_size, self.num_workers = root_dir, batch_size, num_workers

    def setup(self, stage=None):
        # 1. Encode all categorical columns safely
        for df in [self.train_df, self.valid_df]:
            df["State"] = df["State"].apply(lambda x: safe_encode(STATE_LE, x))
            df["target_name"] = df["target_name"].apply(lambda x: safe_encode(TARGET_LE, x))
            df["Species"] = df["Species"].apply(lambda x: safe_encode(SPECIES_LE, x))

        # 2. Calculate Class Weights for Imbalance Handling
        class_counts = Counter(self.train_df["State"])
        num_samples = len(self.train_df)
        class_weights = {c: num_samples / count for c, count in class_counts.items()}
        self.sample_weights = [class_weights[t] for t in self.train_df["State"]]

        # 3. Enhanced Transforms for Plants
        self.train_tf = v2.Compose([
            v2.RandomResizedCrop(300, scale=(0.8, 1.0)), # Larger resolution for state details
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.5),   # Plants look similar upside down (top-down view)
            v2.RandomRotation(degrees=45),  # Stronger rotation for plants
            v2.ColorJitter(brightness=0.1, contrast=0.1),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        self.valid_tf = v2.Compose([
            v2.Resize(320), 
            v2.CenterCrop(300), 
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        self.train_ds = StateDataset(self.train_df, self.root_dir, self.train_tf)
        self.valid_ds = StateDataset(self.valid_df, self.root_dir, self.valid_tf)

    def train_dataloader(self):
        sampler = WeightedRandomSampler(self.sample_weights, len(self.sample_weights), replacement=True)
        return DataLoader(self.train_ds, batch_size=self.batch_size, sampler=sampler, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

# --- Classifier Model ---
class StateClassifier(pl.LightningModule):
    def __init__(self, num_classes, target_dim, species_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        
        # --- LOADING EFFICIENTNET-B6 ---
        weights_path = "/kaggle/input/efficientnet-b6/pytorch/default/1/efficientnet_b6_lukemelas-24a108a5.pth"
        print(f"Loading backbone from: {weights_path}")
        
        try:
            self.base_model = models.efficientnet_b6(weights=None)
            state_dict = torch.load(weights_path, weights_only=True)
            self.base_model.load_state_dict(state_dict, strict=False)
            print("✅ Successfully loaded EfficientNet-B6 weights.")
        except Exception as e:
            print(f"⚠️ Error loading custom weights, using random init: {e}")

        # Remove Head & Get Dim
        self.img_dim = self.base_model.classifier[1].in_features
        self.base_model.classifier = nn.Identity()

        # Embeddings
        self.target_emb = nn.Embedding(target_dim + 1, 8)
        self.species_emb = nn.Embedding(species_dim + 1, 16) # Increased dim for Species info
        
        # Tabular Fusion Layer: Target(8) + Height(1) + Species(16) = 25 inputs
        self.tabular_net = nn.Sequential(
            nn.Linear(25, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Final Classification Head
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(), # Swish
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )
        
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, img, target_name, height, species):
        img_feats = self.base_model(img)
        if len(img_feats.shape) > 2:
            img_feats = img_feats.view(img_feats.size(0), -1)
        
        t_feat = self.target_emb(target_name.long())
        s_feat = self.species_emb(species.long())
        
        # [Target, Height, Species]
        tab_in = torch.cat([t_feat, height.unsqueeze(1), s_feat], dim=1)
        tab_feats = self.tabular_net(tab_in)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1], tab[:, 2])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1], tab[:, 2])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        # Differential Learning Rates
        optimizer = torch.optim.AdamW([
            {'params': self.base_model.parameters(), 'lr': self.lr * 0.1}, # Slow learning for backbone
            {'params': self.tabular_net.parameters(), 'lr': self.lr},
            {'params': self.head.parameters(), 'lr': self.lr}
        ], weight_decay=1e-2)
        
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=[self.lr * 0.1, self.lr, self.lr],
            total_steps=self.trainer.estimated_stepping_batches,
            pct_start=0.3
        )
        return [optimizer], [scheduler]

# --- Training Section ---
train_df, valid_df = train_test_split(train_pd, test_size=0.15, random_state=42, stratify=train_pd["State"])

# Using Batch Size 8 with Accumulation 2 = Effective Batch 16 (Safe for VRAM)
datamodule = StateDataModule(train_df=train_df, valid_df=valid_df, root_dir=image_root_dir, batch_size=8)
datamodule.setup()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state_model = StateClassifier(
    num_classes=len(STATE_LE.classes_), 
    target_dim=len(TARGET_LE.classes_), 
    species_dim=len(SPECIES_LE.classes_),
    lr=5e-4 # Higher LR for OneCycle
).to(device)

early_stop_callback = EarlyStopping(monitor="val_loss", patience=6, mode="min")
checkpoint_callback = ModelCheckpoint(monitor="val_acc", save_top_k=1, mode="max", filename="best-state")

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=15,
    precision="16-mixed",
    accumulate_grad_batches=2,
    callbacks=[early_stop_callback, checkpoint_callback]
)

trainer.fit(state_model, datamodule)

# --- Inference Section (With TTA) ---
print("\n=== Starting State Inference (With TTA) ===")
# Load Best Model
best_model_path = checkpoint_callback.best_model_path
if best_model_path:
    print(f"Loading best state model: {best_model_path}")
    state_model = StateClassifier.load_from_checkpoint(best_model_path)

state_model.eval()
state_model.to(device)

base_tf = v2.Compose([
    v2.Resize(320), v2.CenterCrop(300), v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

state_results = []

with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting State"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img_pil = Image.open(img_path).convert("RGB")
            
            # TTA: Original + Flip
            img_norm = base_tf(img_pil).unsqueeze(0).to(device)
            img_flip = base_tf(img_pil.transpose(Image.FLIP_LEFT_RIGHT)).unsqueeze(0).to(device)
            
            # Prepare Tabular Inputs (Using PREDICTED species from previous step)
            t_tensor = torch.tensor([safe_encode(TARGET_LE, row["target_name"])], device=device)
            h_tensor = torch.tensor([float(row["Height_Ave_cm"])], device=device)
            s_tensor = torch.tensor([safe_encode(SPECIES_LE, row["Species"])], device=device) # Crucial!
            
            # Inference
            logits_norm = state_model(img_norm, t_tensor, h_tensor, s_tensor)
            logits_flip = state_model(img_flip, t_tensor, h_tensor, s_tensor)
            
            # Average Results
            probs = (F.softmax(logits_norm, dim=1) + F.softmax(logits_flip, dim=1)) / 2
            
            class_idx = probs.argmax(dim=1).item()
            state_results.append(STATE_LE.inverse_transform([class_idx])[0])
            
        except Exception as e:
            state_results.append(STATE_LE.classes_[0])

test_pd["State"] = state_results
print("✅ State predictions completed & saved to test_pd")


=== STEP 3: State Prediction (Custom Model: EfficientNet-B6) [Improved] ===
Loading backbone from: /kaggle/input/efficientnet-b6/pytorch/default/1/efficientnet_b6_lukemelas-24a108a5.pth


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.


✅ Successfully loaded EfficientNet-B6 weights.


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ base_model  │ EfficientNet     │ 40.7 M │ train │     0 │
│ 1 │ target_emb  │ Embedding        │     48 │ train │     0 │
│ 2 │ species_emb │ Embedding        │    256 │ train │     0 │
│ 3 │ tabular_net │ Sequential       │  1.8 K │ train │     0 │
│ 4 │ head        │ Sequential       │  1.2 M │ train │     0 │
│ 5 │ loss_fn     │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────┴──────────────────┴────────┴───────┴───────┘

...

Output()

`Trainer.fit` stopped: `max_epochs=15` reached.



=== Starting State Inference (With TTA) ===
Loading best state model: /kaggle/working/lightning_logs/version_1/checkpoints/best-state.ckpt
Loading backbone from: /kaggle/input/efficientnet-b6/pytorch/default/1/efficientnet_b6_lukemelas-24a108a5.pth
✅ Successfully loaded EfficientNet-B6 weights.


Predicting State: 100%|██████████| 5/5 [00:00<00:00,  7.59it/s]

✅ State predictions completed & saved to test_pd


In [8]:
import torch

torch.cuda.empty_cache()

In [9]:
# ============================================================================
# CELL 5: NDVI PREDICTION
# ============================================================================
print("\n=== STEP 4: NDVI Prediction ===")

def image_to_mask(image_path):
    """Applies HSV masking to isolate green vegetation."""
    image = cv2.imread(image_path)
    if image is None:
        return np.zeros((224, 224, 3), dtype=np.uint8)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    
    # Define range for green color
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])
    
    mask = cv2.inRange(hsv, lower_green, upper_green)
    return cv2.bitwise_and(image, image, mask=mask)

class PreGSSHDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row["image_path"])
        
        # 1. Load and Mask Image
        masked_img = image_to_mask(img_path)
        
        # 2. Prepare Tabular Data
        s_idx = safe_encode(SPECIES_LE, row["Species"])
        t_idx = safe_encode(TARGET_LE, row["target_name"])
        ndvi_val = row["Pre_GSHH_NDVI"] if "Pre_GSHH_NDVI" in self.df.columns else 0.0
        
        tabular = torch.tensor([
            float(s_idx),
            float(ndvi_val),
            float(row["Height_Ave_cm"]),
            float(t_idx)
        ], dtype=torch.float32)

        target = torch.tensor([ndvi_val], dtype=torch.float32)

        # 3. Apply Transforms
        if self.transform:
            image = self.transform(masked_img)
        else:
            image = torch.tensor(masked_img).permute(2, 0, 1).float()

        return image, tabular, target

class PreGSSHDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch=16):
        super().__init__()
        self.train_df, self.valid_df = train_df, valid_df
        self.root_dir, self.batch = root_dir, batch
        
        # --- UPDATED TRANSFORMS WITH ROTATION ---
        # Training: Include RandomRotation(180) to cover all angles
        self.train_tf = v2.Compose([
            v2.ToImage(),
            v2.RandomRotation(degrees=180),  # Rotates between -180 and +180 degrees
            v2.Resize((224, 224)),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        # Validation: No rotation, just resize and normalize
        self.valid_tf = v2.Compose([
            v2.ToImage(),
            v2.Resize((224, 224)),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def setup(self, stage=None):
        self.train_ds = PreGSSHDataset(self.train_df, self.root_dir, self.train_tf)
        self.valid_ds = PreGSSHDataset(self.valid_df, self.root_dir, self.valid_tf)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch, shuffle=True, num_workers=2)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch, num_workers=2)

class PreGSSHModel(pl.LightningModule):
    def __init__(self, species_dim, target_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # 1. Backbone with Local Weights
        self.resnet = models.resnet50(weights=None)
        
        # Adjust this path if your dataset location changes
        local_weights = '/kaggle/input/se_resnet50/pytorch/default/1/se_resnet50-ce0d4300.pth'
        
        if os.path.exists(local_weights):
            state_dict = torch.load(local_weights, map_location="cpu")
            # Cleaning keys for standard ResNet50 compatibility
            new_state_dict = {k.replace("layer0.", "").replace("last_linear", "fc"): v for k, v in state_dict.items()}
            self.resnet.load_state_dict(new_state_dict, strict=False)
            print("✅ Loaded SE-ResNet50 local weights")
        else:
            print("⚠️ Local weights not found, using random initialization")
        
        self.img_dim = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        # 2. Categorical Embeddings
        self.species_emb = nn.Embedding(species_dim + 1, 8)
        self.target_emb = nn.Embedding(target_dim + 1, 4)
        
        # 3. Tabular Branch (8 + 4 + 1 height = 13 inputs)
        self.tabular_net = nn.Sequential(
            nn.Linear(13, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # 4. Final Head with Sigmoid
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 1),
            nn.Sigmoid() # Forces output to [0, 1] range
        )
        
        self.loss_fn = nn.HuberLoss(delta=0.1) # Smooth L1/Huber is great for 0-1 ranges

    def forward(self, img, species, target_name, height):
        img_feats = self.resnet(img)
        
        # Embeddings
        s_feat = self.species_emb(species.long())
        t_feat = self.target_emb(target_name.long())
        
        # Combine tabular (Species + Target + Height)
        tab_in = torch.cat([s_feat, t_feat, height.unsqueeze(1)], dim=1)
        tab_feats = self.tabular_net(tab_in)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        # tab order based on Dataset: [species, ndvi, height, target]
        # We pass species(0), target(3), height(2)
        preds = self(img, tab[:, 0], tab[:, 3], tab[:, 2])
        loss = self.loss_fn(preds, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        preds = self(img, tab[:, 0], tab[:, 3], tab[:, 2])
        loss = F.mse_loss(preds, y) # Track MSE for validation
        self.log("val_mse", loss, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=self.hparams.lr, 
            total_steps=self.trainer.estimated_stepping_batches
        )
        return [optimizer], [{"scheduler": scheduler, "interval": "step"}]

# ==========================================
# TRAINING EXECUTION
# ==========================================
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42)

# Initialize DataModule
datamodule = PreGSSHDataModule(train_df, valid_df, image_root_dir)

# Initialize Model
species_count = len(SPECIES_LE.classes_)
target_count = len(TARGET_LE.classes_)
ndvi_model = PreGSSHModel(species_dim=species_count, target_dim=target_count)

# Callbacks
early_stop_callback = EarlyStopping(
    monitor="val_mse",
    patience=5,
    mode="min"
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_mse",
    save_top_k=1,
    mode="min"
)

# Trainer
trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=40,
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback]
)

trainer.fit(ndvi_model, datamodule)

# ==========================================
# INFERENCE EXECUTION
# ==========================================
print("\nStarting Inference...")
ndvi_model.eval()
ndvi_model.to(device)

# Transforms for inference (same as validation)
inf_tfs = v2.Compose([
    v2.ToImage(), 
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

ndvi_results = []

with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting NDVI"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            masked = image_to_mask(img_path)
            img_tensor = inf_tfs(masked).unsqueeze(0).to(device)
            
            # Prepare inputs
            s_idx = torch.tensor([safe_encode(SPECIES_LE, row["Species"])]).to(device)
            t_idx = torch.tensor([safe_encode(TARGET_LE, row["target_name"])]).to(device)
            h_val = torch.tensor([float(row["Height_Ave_cm"])]).to(device)
            
            # Predict
            pred_ndvi = ndvi_model(img_tensor, s_idx, t_idx, h_val).item()
            ndvi_results.append(float(pred_ndvi))
        except Exception as e:
            # Fallback for bad images
            ndvi_results.append(0.0)

test_pd["Pre_GSHH_NDVI"] = ndvi_results
print("✅ NDVI predictions completed")


=== STEP 4: NDVI Prediction ===


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.


✅ Loaded SE-ResNet50 local weights


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ resnet      │ ResNet     │ 23.5 M │ train │     0 │
│ 1 │ species_emb │ Embedding  │    128 │ train │     0 │
│ 2 │ target_emb  │ Embedding  │     24 │ train │     0 │
│ 3 │ tabular_net │ Sequential │  1.0 K │ train │     0 │
│ 4 │ head        │ Sequential │  270 K │ train │     0 │
│ 5 │ loss_fn     │ HuberLoss  │      0 │ train │     0 │
└───┴─────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 23.8 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 23.8 M                                                                                               
Total estimated model params size (MB): 95                                                                         
Modules in train mode: 165                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()


Starting Inference...


Predicting NDVI: 100%|██████████| 5/5 [00:00<00:00,  8.53it/s]

✅ NDVI predictions completed


In [10]:
import torch

torch.cuda.empty_cache()

In [11]:
# ============================================================================
# CELL 6: FINAL BIOMASS PREDICTION
# ============================================================================

print("\n=== STEP 5: Final Biomass Prediction ===")

# Fit the final encoder for target_name if not already done
TARGET_NAME_LE = LabelEncoder()
TARGET_NAME_LE.fit(train_pd["target_name"].astype(str).unique())

class BiomassDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            img = Image.open(image_path).convert("RGB")
        except:
            img = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform:
            img = self.transform(img)

        # Build tabular feature vector
        # Order: [State, Species, NDVI, Height, Target_Name]
        tab = torch.tensor([
            float(safe_encode(STATE_LE, row["State"])),
            float(safe_encode(SPECIES_LE, row["Species"])),
            float(row["Pre_GSHH_NDVI"]),
            float(row["Height_Ave_cm"]),
            float(safe_encode(TARGET_NAME_LE, row["target_name"]))
        ], dtype=torch.float32)

        if self.is_train:
            # The 'target' column is the actual biomass in the training set
            target = torch.tensor(row["target"], dtype=torch.float32)
            return img, tab, target
        else:
            return img, tab

class BiomassLightningModel(pl.LightningModule):
    def __init__(self, state_dim, species_dim, target_name_dim, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        # 1. Load Backbone Weights
        self.resnet = models.resnet50(weights=None)
        
        # Adjust path if needed
        local_weights = '/kaggle/input/se_resnet50/pytorch/default/1/se_resnet50-ce0d4300.pth'
        
        if os.path.exists(local_weights):
            state_dict = torch.load(local_weights, map_location="cpu")
            # Cleaning keys for standard ResNet50
            new_state_dict = {k.replace("layer0.", "").replace("last_linear", "fc"): v for k, v in state_dict.items()}
            self.resnet.load_state_dict(new_state_dict, strict=False)
            print("✅ SE-ResNet50 Weights Loaded Successfully")

        self.img_dim = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        # 2. Define New Layers
        self.state_emb = nn.Embedding(state_dim + 1, 8)
        self.species_emb = nn.Embedding(species_dim + 1, 16)
        self.target_name_emb = nn.Embedding(target_name_dim + 1, 8)
        
        # Tabular Input: 8 (State) + 16 (Species) + 8 (Target) + 2 (NDVI, Height) = 34
        self.tab_net = nn.Sequential(
            nn.Linear(34, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64)
        )
        
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )

        # 3. Initialize Custom Layers
        for m in [self.state_emb, self.species_emb, self.target_name_emb, self.tab_net, self.head]:
            for layer in m.modules():
                if isinstance(layer, (nn.Linear, nn.Conv2d)):
                    nn.init.kaiming_normal_(layer.weight, mode='fan_out', nonlinearity='relu')
                elif isinstance(layer, nn.BatchNorm1d):
                    nn.init.constant_(layer.weight, 1)
                    nn.init.constant_(layer.bias, 0)

    def forward(self, img, state, species, target_name, ndvi_height):
        img_feats = self.resnet(img)
        
        s_emb = self.state_emb(state.long())
        sp_emb = self.species_emb(species.long())
        t_emb = self.target_name_emb(target_name.long())
        
        tab_combined = torch.cat([s_emb, sp_emb, t_emb, ndvi_height], dim=1)
        tab_feats = self.tab_net(tab_combined)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined).squeeze(1)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        
        # Split tab back into components: [State, Species, NDVI, Height, Target_Name]
        state = tab[:, 0]
        species = tab[:, 1]
        ndvi_height = tab[:, 2:4] # Columns 2 and 3
        t_name = tab[:, 4]
        
        preds = self(img, state, species, t_name, ndvi_height)
        loss = F.huber_loss(preds, y) # Huber is robust for regression
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        
        state = tab[:, 0]
        species = tab[:, 1]
        ndvi_height = tab[:, 2:4]
        t_name = tab[:, 4]
        
        preds = self(img, state, species, t_name, ndvi_height)
        
        loss = F.mse_loss(preds, y)
        self.log("val_mse", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.05)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=self.hparams.lr, 
            total_steps=self.trainer.estimated_stepping_batches
        )
        return [optimizer], [{"scheduler": scheduler, "interval": "step"}]

# ==========================================
# TRAINING CONFIGURATION
# ==========================================
train_df, valid_df = train_test_split(train_pd, test_size=0.15, random_state=42)

# --- DEFINING TRANSFORMS WITH ROTATION ---
# Training: Include Rotation to cover all angles
train_tf = v2.Compose([
    v2.RandomRotation(degrees=180), # Rotates -180 to +180 degrees
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Validation/Inference: No rotation
valid_tf = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Create Datasets and Loaders
train_ds = BiomassDataset(train_df, image_root_dir, train_tf, is_train=True)
valid_ds = BiomassDataset(valid_df, image_root_dir, valid_tf, is_train=True)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_ds, batch_size=16, num_workers=2)

# Calculate Dimensions for Embeddings
num_states = len(STATE_LE.classes_)
num_species = len(SPECIES_LE.classes_)
num_targets = len(TARGET_NAME_LE.classes_)

# Initialize Model
biomass_model = BiomassLightningModel(
    state_dim=num_states, 
    species_dim=num_species, 
    target_name_dim=num_targets
)

# Callbacks
early_stop_callback = EarlyStopping(
    monitor="val_mse",  
    patience=5,           
    mode="min",
    verbose=True
)

checkpoint_callback = ModelCheckpoint(
    dirpath="/kaggle/working/checkpoints",
    filename="best-biomass-model",
    monitor="val_mse",
    save_top_k=1,
    mode="min"
)

# Trainer
trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=40,
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback]
)

# Start Training
trainer.fit(biomass_model, train_loader, valid_loader)

# ==========================================
# FINAL INFERENCE
# ==========================================
print("\nStarting Final Inference...")
biomass_model.eval()
biomass_model.to(device)

final_results = []

# Use valid_tf (no rotation) for inference
with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Calculating Biomass"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            
            # Use the validation transform here
            img_tensor = valid_tf(img).unsqueeze(0).to(device)
            
            # Extract values for separate arguments
            s_val = torch.tensor([safe_encode(STATE_LE, row["State"])]).to(device)
            sp_val = torch.tensor([safe_encode(SPECIES_LE, row["Species"])]).to(device)
            t_val = torch.tensor([safe_encode(TARGET_NAME_LE, row["target_name"])]).to(device)
            
            # NDVI and Height combined into one tensor (shape: 1, 2)
            nh_val = torch.tensor([[float(row["Pre_GSHH_NDVI"]), float(row["Height_Ave_cm"])]], dtype=torch.float32).to(device)
            
            # Predict
            pred_biomass = biomass_model(img_tensor, s_val, sp_val, t_val, nh_val).item()
            
            final_results.append(max(0.0, pred_biomass))
        except Exception as e:
            # Fallback to mean or 0 if image fails
            final_results.append(0.0)

# ==========================================
# CREATE SUBMISSION
# ==========================================
submission_df = pd.DataFrame({
    "sample_id": test_pd_original["sample_id"],
    "target": final_results
})

submission_df.to_csv('/kaggle/working/submission.csv', index=False)
print(f"\n✅ Pipeline Complete! Submission saved to submission.csv")
print(submission_df.head(10))


=== STEP 5: Final Biomass Prediction ===


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loading `train_dataloader` to estimate number of stepping batches.


✅ SE-ResNet50 Weights Loaded Successfully


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ resnet          │ ResNet     │ 23.5 M │ train │     0 │
│ 1 │ state_emb       │ Embedding  │     40 │ train │     0 │
│ 2 │ species_emb     │ Embedding  │    256 │ train │     0 │
│ 3 │ target_name_emb │ Embedding  │     48 │ train │     0 │
│ 4 │ tab_net         │ Sequential │ 13.0 K │ train │     0 │
│ 5 │ head            │ Sequential │  1.1 M │ train │     0 │
└───┴─────────────────┴────────────┴────────┴───────┴───────┘

...

Output()

Metric val_mse improved. New best score: 992.440
Metric val_mse improved by 639.537 >= min_delta = 0.0. New best score: 352.903
Metric val_mse improved by 197.441 >= min_delta = 0.0. New best score: 155.461
Monitored metric val_mse did not improve in the last 5 records. Best score: 155.461. Signaling Trainer to stop.



Starting Final Inference...


Calculating Biomass: 100%|██████████| 5/5 [00:00<00:00, 15.07it/s]



✅ Pipeline Complete! Submission saved to submission.csv
                    sample_id      target
0  ID1001187975__Dry_Clover_g  149.536865
1    ID1001187975__Dry_Dead_g  175.833466
2   ID1001187975__Dry_Green_g  241.645264
3   ID1001187975__Dry_Total_g  269.716614
4         ID1001187975__GDM_g  260.142365


In [12]:
submission_df.head()

,sample_id,target
0,ID1001187975__Dry_Clover_g,149.536865
1,ID1001187975__Dry_Dead_g,175.833466
2,ID1001187975__Dry_Green_g,241.645264
3,ID1001187975__Dry_Total_g,269.716614
4,ID1001187975__GDM_g,260.142365
